In [1]:
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input
import os
import numpy as np
from tensorflow.keras.utils import load_img, img_to_array
from sklearn.model_selection import train_test_split

In [2]:
model = ResNet50(
    weights="imagenet",
    include_top=False,
    pooling="avg",
    input_shape=(224, 224, 3)
)

model.trainable = False

In [3]:
def extract_embedding(img_path):
    img = load_img(img_path, target_size=(224, 224))
    img = img_to_array(img)
    img = np.expand_dims(img, axis=0)
    img = preprocess_input(img)
    emb = model(img, training=False).numpy()[0]
    return emb

In [ ]:
DATA_DIR = "../data/UTKFace"

def parse_utk_filename(fname):
    parts = fname.split("_")
    if len(parts) < 3:
        return None
    try:
        age = int(parts[0])
        gender = int(parts[1])
        race = int(parts[2])
        if gender not in (0, 1) or race not in range(5) or age < 0 or age > 120:
            return None
        return age, gender, race
    except ValueError:
        return None

embeddings = []
ages = []
genders = []
races = []

files = sorted(os.listdir(DATA_DIR))
total = len(files)
print(f"Total files found: {total}")

for i, fname in enumerate(files):
    labels = parse_utk_filename(fname)
    if labels is None:
        continue
    age, gender, race = labels
    fpath = os.path.join(DATA_DIR, fname)
    try:
        emb = extract_embedding(fpath)
        embeddings.append(emb)
        ages.append(age)
        genders.append(gender)
        races.append(race)
    except Exception as e:
        print(f"Skipping {fname}: {e}")
    if (i + 1) % 500 == 0:
        print(f"Processed {i + 1}/{total} files...")

embeddings = np.array(embeddings)
ages = np.array(ages)
genders = np.array(genders)
races = np.array(races)

print(f"\nTotal valid samples: {len(embeddings)}")
print(f"Embedding shape: {embeddings.shape}")
print(f"Age range: {ages.min()} - {ages.max()}")
print(f"Gender distribution: {np.bincount(genders)}")
print(f"Race distribution: {np.bincount(races)}")

Total files found: 23708
Processed 500/23708 files...
Processed 1000/23708 files...
Processed 1500/23708 files...
Processed 2000/23708 files...
Processed 2500/23708 files...
Processed 3000/23708 files...
Processed 3500/23708 files...
Processed 4000/23708 files...
Processed 4500/23708 files...
Processed 5000/23708 files...
Processed 5500/23708 files...
Processed 6000/23708 files...
Processed 6500/23708 files...
Processed 7000/23708 files...
Processed 7500/23708 files...
Processed 8000/23708 files...
Processed 8500/23708 files...
Processed 9000/23708 files...
Processed 9500/23708 files...
Processed 10000/23708 files...
Processed 10500/23708 files...
Processed 11000/23708 files...
Processed 11500/23708 files...
Processed 12000/23708 files...
Processed 12500/23708 files...
Processed 13000/23708 files...
Processed 13500/23708 files...
Processed 14000/23708 files...
Processed 14500/23708 files...
Processed 15000/23708 files...
Processed 15500/23708 files...
Processed 16000/23708 files...
Pro

In [4]:
# -- Save data from the UTK dataset --
strat_key = genders * 5 + races

X_train, X_test, y_age_train, y_age_test, y_gender_train, y_gender_test, y_race_train, y_race_test = \
    train_test_split(embeddings, ages, genders, races, test_size=0.2, random_state=42, stratify=strat_key)

print(f"Train: {X_train.shape}, Test: {X_test.shape}")

np.save("X_train_embeddings_UTK.npy", X_train)
np.save("X_test_embeddings_UTK.npy", X_test)

np.save("y_train_labels_UTK.npy", np.column_stack([y_age_train, y_gender_train, y_race_train]))
np.save("y_test_labels_UTK.npy", np.column_stack([y_age_test, y_gender_test, y_race_test]))

print("Saved embeddings and labels (columns: age, gender)")

Train: (18964, 2048), Test: (4741, 2048)
Saved embeddings and labels (columns: age, gender, race)
